# all

In [1]:
# Parse the arguments
upper_thresh = 1000
max_lower_thresh = 150
thresh = 50
video_path = 'src/clock/clock.mp4'
box_file_path =  "data.csv"
gif_file_path = "vis.GIF"

In [2]:
#python 3.9.2
#use conda virtual environment, no specific requirement for libraries' versions

import os
from glob import glob
import re
import numpy as np
import cv2
from PIL import Image

import shutil
from utils import *

# Call the function to convert the video to images
convert_video_to_images(video_path, 'Frame')

# data_path is the folder name without / at the end
# assume the folder contains only a numerical-ordered number of images ended with jpg
# e.g., ['10s_Frame/frame_0.jpg', '10s_Frame/frame_1.jpg', '10s_Frame/frame_2.jpg']

data_path = 'Frame' 
image_paths = sorted(glob(f"{data_path}/*.jpg"), key=lambda x: int(re.search(r'(\d+).jpg', os.path.basename(x)).group(1)))

kernel=np.array((9,9), dtype=np.uint8)

box_result = []
for idx in range(1, len(image_paths)):
    # read frames
    frame1_bgr = cv2.imread(image_paths[idx - 1])
    frame2_bgr = cv2.imread(image_paths[idx])

    # get detections
    detections = get_detections(cv2.cvtColor(frame1_bgr, cv2.COLOR_BGR2GRAY), 
                                cv2.cvtColor(frame2_bgr, cv2.COLOR_BGR2GRAY), 
                                bbox_thresh=thresh,
                                nms_thresh=1e-4)
    box_result.append(detections)
print('the box thresh: ', thresh)  
'''
# example box output: list of array: an array is for a image, an list in the array is for a box; 
# box format: [xmin, ymin, xmax, ymax]
[array([[ 767,  596,  926,  750],
        [ 130,  372,  248,  449],
        [1142,  195, 1223,  267],
        [ 701,  127,  779,  180],
        [ 845,   34,  911,   95],
        [ 995,    0, 1053,   41]]),
 array([[ 766,  599,  926,  755],
        [ 135,  370,  255,  447],
        [1142,  196, 1222,  269],
        [ 846,   29,  913,   93],
        [ 705,  125,  781,  178],
        [ 997,    0, 1055,   39]])]
'''

# Save the list of arrays to a CSV file

# Check if the file exists before attempting to delete it
if os.path.exists(box_file_path):
    os.remove(box_file_path)
    print(f"{box_file_path} has been deleted.")
else:
    print(f"{box_file_path} does not exist. Create a new one")


with open(box_file_path, 'w') as file:
    for idx, arr in enumerate(box_result):
        np.savetxt(file, arr, delimiter=',', fmt='%d')
        if idx < len(box_result) - 1:
            file.write(';\n')  # Add a ; between arrays, except for the last one
print('the output file: ', box_file_path)        
'''
# how to import the csv
import numpy as np

# Initialize an empty list to store arrays
box_result = []

# Read the CSV file
with open('data.csv', 'r') as file:
    data = file.read().split(';\n')  # Split the file content by the word 'WORD'

    # Process each part of the split content
    for part in data:
        if part.strip():  # Check if the part is not empty
            arr = np.genfromtxt(part.splitlines(), delimiter=',', dtype=int)
            #box_result.append(arr)
        else:
            arr = np.zeros((0, 4))#'No Diff'
        box_result.append(arr)
# Print the list of arrays
for arr in box_result:
    print(arr)

'''


############################################


#visualize
from PIL import Image
import matplotlib.pyplot as plt


def draw_bboxes(frame, detections):
    for det in detections:
        x1,y1,x2,y2 = det
        area = (x2-x1)*(y2-y1)
        if area > upper_thresh:
            cv2.rectangle(frame, (x1,y1), (x2,y2), (0,255,0), 3) # green
        elif area < max_lower_thresh:
            cv2.rectangle(frame, (x1,y1), (x2,y2), (255,0,0), 3) # red
        #cv2.rectangle(frame, (x1,y1), (x2,y2), (0,255,0), 3) #(0,255,0), 3) green


#'''
# Create output folder if it doesn't exist
import shutil
if not os.path.exists('temp'):
    os.makedirs('temp')
else:
    shutil.rmtree('temp')
    os.makedirs('temp')
    
print('start to visualize the box on the images')

for idx in range(1, len(image_paths)):
    # read frames
    frame_bgr = cv2.imread(image_paths[idx])
    detections = box_result[idx-1]                    
    # draw bounding boxes on frame
    draw_bboxes(frame_bgr, detections)

    # save image for GIF
    fig = plt.figure(figsize=(15, 7))
    plt.imshow(frame_bgr)
    plt.axis('off')
    fig.savefig(f"temp/frame_{idx}.png")
    plt.close()

# Check if the file exists before attempting to delete it
if os.path.exists(gif_file_path):
    os.remove(gif_file_path)
    print(f"{gif_file_path} has been deleted.")
else:
    print(f"{gif_file_path} does not exist.")

create_gif_from_images(gif_file_path, 'temp', '.png')

#'''

Output folder already exists: Frame
Recreating output folder: Frame
Total frames: 335
the box thresh:  50
data.csv has been deleted.
the output file:  data.csv
start to visualize the box on the images
vis.GIF has been deleted.
start to create GIF
